# 19장 · 방법과 결과 절 쓰는 법

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [19장 · 방법과 결과 절 쓰는 법](https://grow.minds.kr/textbooks/css-methods/causal/book/ch19-결과-절-쓰는-법.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 구간은 무엇의 구간인가
*d* 의 구간과 **평균 차이**의 구간은 다른 수다. 섞어 적는 것이 이 장의 대표 오류.

In [ ]:
exp = load("exp")
x = exp[exp.cond == 1].mil.values; y = exp[exp.cond == 0].mil.values
n1, n2 = len(x), len(y)
sp = np.sqrt(((n1-1)*x.var(ddof=1) + (n2-1)*y.var(ddof=1)) / (n1+n2-2))
d = (x.mean() - y.mean()) / sp
se_d = np.sqrt((n1+n2)/(n1*n2) + d**2 / (2*(n1+n2)))
print("d 와 그 구간 :", round(d,3), round(d-1.96*se_d,3), round(d+1.96*se_d,3))   # 0.231 0.025 0.436
tcrit = stats.t.ppf(.975, n1+n2-2); se_diff = sp*np.sqrt(1/n1 + 1/n2)
md = x.mean() - y.mean()
print("평균차와 그 구간:", round(md,3), round(md-tcrit*se_diff,3), round(md+tcrit*se_diff,3))  # 0.283 0.031 0.535

## 3. 오차 막대 세 종
같은 자료인데 폭이 열 배 넘게 벌어진다. 캡션에 **무엇인지** 안 적으면 그림이 거짓말을 한다.

In [ ]:
print("SD:", round(x.std(ddof=1),3),
      " SE:", round(x.std(ddof=1)/np.sqrt(n1),3),
      " 95%CI 반폭:", round(tcrit*x.std(ddof=1)/np.sqrt(n1),3))   # 1.18 0.087 0.171

## 4. 결과 절 수치를 한 번에
손으로 옮겨 적지 않는다. 함수 하나가 본문의 수를 전부 다시 뽑는다.

In [ ]:
def 보고(x, y, name):
    n1, n2 = len(x), len(y)
    sp = np.sqrt(((n1-1)*x.var(ddof=1) + (n2-1)*y.var(ddof=1)) / (n1+n2-2))
    d = (x.mean() - y.mean()) / sp
    t = (x.mean() - y.mean()) / (sp * np.sqrt(1/n1 + 1/n2)); df = n1 + n2 - 2
    p = 2 * stats.t.sf(abs(t), df)
    se_d = np.sqrt((n1+n2)/(n1*n2) + d**2/(2*(n1+n2)))
    print(f"{name}: M {x.mean():.2f}({x.std(ddof=1):.2f}) 대 {y.mean():.2f}({y.std(ddof=1):.2f}) "
          f"t({df})={t:.2f} p={p:.3f} d={d:.2f} CI[{d-1.96*se_d:.2f}, {d+1.96*se_d:.2f}]")
for v in ("mil", "hjs", "flr", "age", "mil_t1"):
    보고(exp[exp.cond==1][v].values, exp[exp.cond==0][v].values, v)

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.